# LLM Data Engineering

> 15 trillion tokens -- the training data size of LLaMA 3. The number sounds enormous, but where does it come from? It certainly does not fall from the sky.
>
> This section starts from raw HTML on the internet and walks through the complete data cleaning pipeline: text extraction, quality filtering, deduplication, and data mixing -- why each step is needed and how it is done. We finish by examining why over 70% of post-training data is synthetic.

The starting point of data engineering is Common Crawl -- a non-profit project that crawls the entire web every month, with raw data volume around 500TB/month. This data is in HTML format, mixed with navigation bars, ads, JavaScript code, and comment sections.

Feeding it directly to a model is equivalent to feeding it garbage; we need a systematic method to turn raw HTML into clean, trainable text. The whole process has five steps: text extraction -> quality filtering -> deduplication -> data mixing -> Tokenize. The filtering criteria at each step directly affect the final model quality, so each step needs clear decision rules.

This section breaks down the five steps in order, using real data fragments to show the changes before and after filtering at each step.

## 0. What Pretraining Data Contains

| Type | Representative Source | Rough Share | Contribution |
|:---|:---|:---|:---|
| Web | filtered Common Crawl, FineWeb | usually 60%+ | broad knowledge and language diversity |
| Encyclopedias | Wikipedia | about 1% | dense factual knowledge |
| Books | Gutenberg, ebooks | several percent | long-form coherence |
| Code | GitHub, The Stack | about 5–15% | logic and strict structure |
| Papers | arXiv | about 2% | scientific terminology and reasoning |
| Mathematics | OpenWebMath-style corpora | about 2% | mathematical reasoning |
| Dialogue | forums and Q&A | several percent | conversational style |

Code improves more than programming because its causal structure and syntax teach disciplined reasoning. Small, high-quality sources can contribute disproportionately and may be repeated more often in a mixture. Synthetic mathematics, instruction, and code data have also become major components.

The recurring lesson is **quality matters more than raw volume**. T5's cleaned C4 and later textbook-style datasets showed that much smaller curated corpora can outperform larger noisy ones. Llama generations likewise improved heavily through data scale, filtering, and mixture changes even when the basic architecture changed little.


## 1. Inspecting Popular Dataset Structures

| Stage | Data Shape | Scale | Examples |
|:---|:---|:---|:---|
| Pretraining | plain documents / Token streams | trillions of Tokens | FineWeb-Edu, RedPajama |
| SFT | instruction → answer or messages | thousands to millions | Alpaca, Tulu-3, COIG |
| Preference training | chosen/rejected answers to one prompt | tens to hundreds of thousands | HH-RLHF, UltraFeedback |

### 1.1 Pretraining: One Document per Record

Representative corpora include FineWeb-Edu (filtered educational web), RedPajama and SlimPajama (LLaMA-style mixtures), The Stack v2 (multilingual code), SkyPile and MAP-CC (Chinese web), Wikipedia, arXiv, and books. Records normally contain a `text` field plus metadata such as source, URL, language score, and timestamp. Training later tokenizes and concatenates these documents.


In [ ]:
# === Open a real pretraining dataset: a 10,000-document sample from The Pile ===
# Download NeelNanda/pile-10k, an approximately 33 MB official sample, rather than simulated data
# Outside mainland China, replace hf-mirror.com with huggingface.co
import os
import requests
import pandas as pd

url = ("https://hf-mirror.com/datasets/NeelNanda/pile-10k/resolve/main/"
       "data/train-00000-of-00001-4746b8785c874cc7.parquet")
cache = "pile-10k.parquet"

if not os.path.exists(cache):
    print("First run: downloading the real sample file, about 33 MB ...")
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    # Verify completeness because a mirror can occasionally truncate a file
    expected = int(r.headers.get("Content-Length", 0))
    assert expected and len(r.content) == expected, "Download incomplete; rerun this cell"
    with open(cache, "wb") as f:
        f.write(r.content)
print(f"Ready: {cache}, {os.path.getsize(cache)/1e6:.1f} MB")

docs = pd.read_parquet(cache)
print(f"Loaded {len(docs)} documents with columns {list(docs.columns)}")
print("The published jsonl format stores one document per line: {text: ..., meta: {...}}")
print()

# Document 0 is a Pile-CC web blog; document 68 is GitHub code
for i in [0, 68]:
    row = docs.iloc[i]
    print(f"--- Document {i}, source {row['meta']['pile_set_name']} ---")
    print("text:", str(row["text"])[:110].replace("\n", " ") + "……")
    print("meta:", row["meta"])
    print()

print("--- Source subsets in meta, top eight ---")
print(docs["meta"].map(lambda m: m["pile_set_name"]).value_counts().head(8))
print()
print("Key observation:")
print("  One line is one complete document: body text in text and auxiliary fields in meta")
print('TODO: replace this placeholder with your code')
print("  Web corpora such as FineWeb also record language_score and perplexity in meta")


### 1.2 SFT and Preference Data: Three Common Formats

SFT datasets include Alpaca-52k, ShareGPT conversations, OASST1 message trees, LIMA's 1,000 curated examples, OpenHermes, Tulu mixtures, BELLE/COIG, and Infinity-Instruct. Common schemas are Alpaca's `instruction/input/output`, chat `messages`, and prompt/completion pairs.

Preference datasets such as HH-RLHF and UltraFeedback store a prompt with `chosen` and `rejected` responses, sometimes with scalar or rubric scores. Before training, normalize role names, dialogue order, system prompts, and answer fields into one internal schema rather than writing a separate pipeline for every source.


In [ ]:
# === Three SFT data formats: Alpaca, ShareGPT, and preference pairs ===
import json

# Format 1: Alpaca uses three flat fields for a single turn
alpaca_sample = {
    "instruction": "Translate the following sentence into English.",
    "input": "The weather is lovely today.",
    "output": "The weather is nice today.",
}

# Format 2: ShareGPT uses a conversations array that naturally supports multiple turns
sharegpt_sample = {
    "conversations": [
        {"from": "human", "value": "Write bubble sort for me"},
        {"from": "gpt", "value": "Certainly. Here is a Python version:\n```python\n...\n```"},
        {"from": "human", "value": "Can you add comments?"},
        {"from": "gpt", "value": "Of course. Here is the commented version ..."},
    ]
}

# Format 3: preference data stores a chosen and rejected response for RLHF or DPO
preference_sample = {
    "prompt": "Briefly explain overfitting",
    "chosen": "Overfitting means a model performs well on training data but worse on new data ...",
    "rejected": "Overfitting means fitting too much, which is bad.",
}

print("=== Three KL Estimators Compared ===")
print()
for name, sample in [("Alpaca, single-turn instruction", alpaca_sample),
                     ("ShareGPT, multi-turn dialogue", sharegpt_sample),
                     ("Preference pair", preference_sample)]:
    print(f"--- {name} ---")
    print(json.dumps(sample, ensure_ascii=False, indent=2))
    print()

print("Key observation:")
print("  Alpaca has one question and answer; instruction plus input forms the prompt")
print("  ShareGPT represents multiple turns in an array and uses from to distinguish roles")
print("  Preference data has no single gold answer; it stores better and worse responses to one prompt")


## 2. Practical Filtering: From Rules to Data-Juicer

Clean dataset rows begin as noisy HTML inside WARC archives. We now implement extraction, filtering, deduplication, and a reusable Data-Juicer pipeline.

### 2.1 Extracting Main Text from HTML

Web pages contain navigation, ads, JavaScript, CSS, footers, and comments around the article. Extraction identifies the main body and discards boilerplate. Production tools such as trafilatura and resiliparse combine HTML structure with learned heuristics. The next cell demonstrates the central transformation on a small page.


In [ ]:
# === Simulation: A typical Common Crawl HTML page ===
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>

  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>

  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>

    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>

    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>

  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>

  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print("=== Raw HTML ===")
print(raw_html[:500])
print("...")
print()
print("^ Mixed inside: nav bar, ads, JS tracking code, comment spam, footer")

In [ ]:
import re
# === Manually simulate text extraction ===
print("=== Text extraction: HTML to plain text ===")
print()

# Step 1: remove script and style tags together with their contents
def remove_scripts_styles(html):
    """Remove script and style tags, including their contents."""
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Step 2: remove every remaining HTML tag
def strip_tags(html):
    """Remove every <...> HTML tag."""
    return re.sub(r'<[^>]+>', '', html)

# Step 3: standard deviation
def clean_whitespace(text):
    """Collapse repeated whitespace and trim the ends."""
    text = re.sub(r'[ \t]+', ' ', text)  # Collapse consecutive spaces and tabs
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  # Reduce many blank lines to one
    return text.strip()

# Walk through the pipeline one step at a time
print("Step 1: remove script and style tags")
no_scripts = remove_scripts_styles(raw_html)
print(f"  Raw: {len(raw_html)} characters; after removal: {len(no_scripts)}")
print()

print("Step 2: remove all HTML tags")
plain_text = strip_tags(no_scripts)
print(f"  Plain-text length: {len(plain_text)} characters")
print()

print("Step 3: normalize whitespace")
clean_text = clean_whitespace(plain_text)
print(f"  Cleaned length: {len(clean_text)} characters")
print()

print("=" * 60)
print("Extracted result:")
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print("Note that navigation, advertisements, and comments remain")
print("      These require the later quality-filtering stage")
print()
print("Data-Juicer counterpart: clean_html_mapper, with tools such as trafilatura in production")


### 2.2 Heuristic Quality Filtering

Extracted text still includes SEO pages, gibberish, empty templates, and repeated advertising. Heuristic filters inspect length, alphabetic ratio, line statistics, symbol density, repeated n-grams, and suspicious boilerplate. Filtering is a precision–recall trade-off: overly weak rules retain noise, while overly aggressive rules erase useful domains and language varieties.


In [ ]:
import numpy as np
np.random.seed(42)
# === Demonstrate quality-filtering rules one by one ===
print("=== Summary table ===")
print()

# Prepare several texts for review
samples = [
    {"name": "good article", "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": "advertisement", "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": "table of contents", "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": "too short", "text": "Hello world."},
    {"name": "random characters", "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": "repeated template", "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    """Return whether to keep the text and, if not, the rejection reason."""
    
# Rule 1: length filter
    words = text.split()
    if len(words) < 5:
        return False, f"too short, {len(words)} words < 5"
    if len(text) > 5000:
        return False, f"too long, {len(text)} characters > 5000"
    
# Rule 2: average word length; ordinary English words are usually 3-10 characters
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, f"abnormal average word length, {avg_word_len:.1f} > 12"
    
# Rule 3: special-character ratio; normal prose rarely exceeds 15%
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, f"too many special characters, {special_ratio:.1%} > 25%"
    
# Rule 4: repeated-line ratio; template pages often repeat the same lines
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, f"too many repeated lines, unique ratio {unique_ratio:.1%} < 40%"
    
# Rule 5: uppercase ratio; text dominated by CAPS LOCK is usually low quality
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, f"too many uppercase letters, {upper_ratio:.1%} > 50%"
    
    return True, "pass"


print(f"{'Text':<12s} {'Result':>8s} {'Reason'}")
print("-" * 55)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = "keep" if passed else "discard"
    print(f"{sample['name']:<12s} {status:>8s}  {reason}")

print()
print("Real systems commonly use 20-50 rules like these.")
print("Together they can remove roughly 60-80% of Common Crawl text.")
print()
print("Data-Juicer counterparts include language_id_score_filter and word_repetition_filter.")


### 2.3 Perplexity-Based Quality Scoring

A lightweight language model such as KenLM can score documents. Very low Perplexity may indicate repetitive template text; ordinary human prose occupies a middle range; extremely high Perplexity often signals corruption. Some pipelines train a classifier with Wikipedia-like text as positive examples and random web pages as negatives. Wikipedia supplies a convenient reference style, not a universal definition of quality.


In [ ]:
# === Visualize the perplexity interval retained by filtering ===
# Simulate KenLM perplexity distributions for web prose and junk; these are teaching values, not measurements
import matplotlib.pyplot as plt

np.random.seed(42)

# Human prose tends to have perplexity between 10 and 1,000
# Junk has two tails: formulaic templates are too predictable, while gibberish is too unpredictable
ppl_human = np.random.lognormal(mean=4.6, sigma=0.8, size=5000)   # Median about 100
ppl_junk_low = np.random.lognormal(mean=1.5, sigma=0.7, size=2500)   # About 4.5
ppl_junk_high = np.random.lognormal(mean=8.2, sigma=0.9, size=2500)  # About 3,600

fig, ax = plt.subplots(figsize=(7, 4.2))
bins = np.logspace(0, 5, 60)
ax.hist(ppl_junk_low, bins=bins, color='#f87171', alpha=0.6, label='Template / boilerplate')
ax.hist(ppl_junk_high, bins=bins, color='#fbbf24', alpha=0.6, label='Garbled / spam text')
ax.hist(ppl_human, bins=bins, color='#2563eb', alpha=0.6, label='Human-written text')
ax.axvspan(10, 1000, color='#16a34a', alpha=0.10, label='Keep band [10, 1000]')
ax.set_xscale('log')
ax.set_xlabel('Perplexity (KenLM)')
ax.set_ylabel('Number of documents')
ax.set_title('PPL filter: keep the middle, drop both tails')
ax.legend(fontsize=8)
plt.show()

print("Key observation: retain the green middle-perplexity interval.")
print("  Left tail: template and list pages are too predictable and add little new information")
print("  Right tail: gibberish and spliced text are hard to model and teach little")
print("  Thresholds 10 and 1,000 are calibrated on samples and downstream evaluations, not derived theoretically")
print()
print("Data-Juicer counterpart: perplexity_filter, which combines KenLM scoring with thresholds")


### 2.4 PII and Safety Filtering

Readable text may still contain personal identifiers, API keys, private keys, passwords, malicious scripts, or unsafe material. This is separate from prose quality.

| Action | Use Case |
|:---|:---|
| Drop document | severe leakage such as keys or identity numbers |
| Redact and keep | valuable body with limited PII; replace with `<EMAIL>` or `<PHONE>` |

Choose by risk level: high-risk documents are removed; low-risk identifiers can be redacted when the surrounding text remains valuable.


In [ ]:
import re

# === Demonstrate PII and security filtering ===
print("=== Summary table ===")
print()

records = [
    {
        "name": "ordinary tutorial",
        "text": "This tutorial explains gradient descent with a small example.",
    },
    {
        "name": "contains email",
        "text": "Contact me at alice@example.com for the dataset notes.",
    },
    {
        "scenario": "Deploying to mobile",
        "text": 'TODO: replace this placeholder with your code',
    },
    {
        "name": "suspected secret key",
        "text": "AWS_SECRET_ACCESS_KEY = ABCDEFGHIJKLMNOPQRSTUVWXYZ1234567890AB",
    },
]

patterns = {
    "EMAIL": r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}",
    "PHONE_CN": r"1[3-9]\d{9}",
    "SECRET": r"(api[_-]?key|secret[_a-z0-9-]*|token|password)\s*[=:]\s*[^\s]+",
}


def redact_or_drop(text):
    """Detect PII or secrets and return the action plus processed text."""
    lowered = text.lower()
    if re.search(patterns["SECRET"], lowered):
        return "discard", "suspected secret key detected"

    cleaned = text
    hit = False
    for label, pattern in patterns.items():
        if label == "SECRET":
            continue
        if re.search(pattern, cleaned):
            hit = True
            cleaned = re.sub(pattern, f"<{label}>", cleaned)

    if hit:
        return "redact and keep", cleaned
    return "keep", cleaned


for record in records:
    action, result = redact_or_drop(record["text"])
    print(f"{record['name']:<8s} -> {action}")
    print(f"  {result}")

print()
print("Key observation: PII and security filters address leakage risk, not prose quality.")
print()
print("Data-Juicer counterparts: clean_email_mapper and clean_phone_mapper")


### 2.5 Deduplication: Exact Hashes and MinHash

Repeated news syndication, copied code, placeholders, and cookie notices waste compute, distort source weights, and encourage memorization.

Use two layers:

1. exact hashes remove byte-identical or normalized-identical documents cheaply;
2. MinHash + locality-sensitive hashing approximates n-gram Jaccard similarity and removes near duplicates such as mirrored pages.

Thresholds should be measured on sampled pairs because aggressive deduplication can remove legitimately related documents.


In [ ]:
import hashlib
# === Exact deduplication ===
print("=== Layer 1: Exact Dedup ===")
print()

# Simulate 5 articles, 2 of which are duplicates
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  # identical to doc 1
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  # identical to doc 2
]

print(f"Total {len(docs)} articles")
print()

# Hash deduplication
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # SHA256 hash: turn the article into a unique fingerprint
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  # show only first 16 chars
    is_new = fingerprint not in seen

    print(f"Doc {i+1}: hash={fingerprint}  {'KEEP ✅' if is_new else 'DUPLICATE, drop'}")

    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print(f"After dedup: {len(unique_docs)} articles ({len(docs) - len(unique_docs)} removed)")
print()
print("Exact dedup removes about 5-15% of Common Crawl data")
print("But that is not enough -- most duplication is 'paraphrase' rather than 'verbatim copy'")

In [ ]:
import hashlib
# === Approximate deduplication with MinHash: hand-calculate the idea ===
print("=== Layer 2: approximate deduplication with MinHash ===")
print()
print("Problem: pairwise comparison among ten billion documents requires 10B squared comparisons")
print()
print("MinHash gives each document a fingerprint; similar fingerprints suggest similar documents")
print()

# === Manual MinHash demonstration ===
print("=== Hand-calculating MinHash ===")
print()

# Three documents
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  # Differs from A by one word
doc_C = "quantum mechanics describes behavior of subatomic particles"  # Unrelated topic

print(f"Document A: {doc_A}")
print(f"Document B: {doc_B}")
print(f"Document C: {doc_C}")
print()

# Step 1: split each document into a set of three-word n-grams
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print(f"Document A n-grams, {len(A_ngrams)} total: {A_ngrams}")
print()
print(f"Document B n-grams, {len(B_ngrams)} total: {B_ngrams}")
print()

# Step 2: calculate Jaccard similarity, intersection divided by union
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print(f"Size of A intersect B: {len(A_ngrams & B_ngrams)}")
print(f"Size of A union B: {len(A_ngrams | B_ngrams)}")
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Step 3: MinHash fingerprints, using random hashes in this simplified demonstration
print("=== Calculate MinHash signatures ===")
print()

# Hash each n-gram to an integer and use several minimum hash values as the signature
def minhash_signature(ngrams, num_hashes=4):
    """Generate a MinHash signature for an n-gram set using several seeded hashes."""
    sig = []
    for i in range(num_hashes):
        # Hash every n-gram with seed i and retain the minimum value
        min_val = float('inf')
        for ng in ngrams:
            raw = (ng + str(i)).encode()
            h = int(hashlib.sha256(raw).hexdigest(), 16) % 100000
            min_val = min(min_val, h)
        if min_val != float('inf'):
            sig.append(min_val)
        else:
            sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print(f"MinHash signature of A: {sig_A}")
print(f"MinHash signature of B: {sig_B}")
print(f"MinHash signature of C: {sig_C}")
print()

# MinHash similarity is the fraction of matching signature entries
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print(f"Approximate MinHash A-B: {mh_AB:.2%}, exact Jaccard: {j_AB:.2%}")
print(f"Approximate MinHash A-C: {mh_AC:.2%}, exact Jaccard: {j_AC:.2%}")
print()
print("MinHash changes pairwise n-gram comparison into comparison of four numbers")
print("This is orders of magnitude faster; above a threshold such as 80%, keep only one document")
print()
print("Data-Juicer counterpart: document_minhash_deduplicator, which uses MinHash plus LSH")


In [ ]:
# === Visualize MinHash estimates versus exact Jaccard similarity ===
import matplotlib.pyplot as plt

pair_labels = ['A vs B\nnear-duplicate', 'A vs C\ndifferent topics']
exact_scores = [j_AB * 100, j_AC * 100]
est_scores = [mh_AB * 100, mh_AC * 100]

x = np.arange(len(pair_labels))
width = 0.32

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(x - width/2, exact_scores, width, color='#2563eb', label='Exact Jaccard')
ax.bar(x + width/2, est_scores, width, color='#f59e0b',
       label='MinHash estimate (4 hashes)')
ax.axhline(80, color='#dc2626', linestyle='--', linewidth=1.5,
           label='Dedup threshold 80%')
for xi, v in zip(x - width/2, exact_scores):
    ax.text(xi, v + 2, f'{v:.0f}%', ha='center', fontsize=9)
for xi, v in zip(x + width/2, est_scores):
    ax.text(xi, v + 2, f'{v:.0f}%', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(pair_labels)
ax.set_ylabel('Similarity (%)')
ax.set_ylim(0, 105)
ax.set_title('4 hashes give a rough estimate; production uses 128-256')
ax.legend(fontsize=8)
plt.show()

print("Key observation: with only four hashes, the estimate can differ substantially from exact Jaccard.")
print("  Production systems use 128-256 hashes, bringing the estimate closer to the true value")
print("  Each document then stores only a few hundred integers instead of its full n-gram set")


### 2.6 Benchmark Decontamination

Internal deduplication removes repetition, but training data may also contain benchmark questions and answers. This leakage inflates scores without improving transferable capability.

| Comparison | Goal | Methods |
|:---|:---|:---|
| training vs training | reduce repetition | hashes, MinHash |
| training vs benchmark | preserve evaluation validity | exact match, n-gram overlap, Embedding similarity |

Decontamination makes evaluation trustworthy rather than merely making text look clean.


In [ ]:
# === Decontamination demo: have benchmark questions leaked into the training data ===
print("=== Decontamination: Training set vs Benchmark ===")
print()

train_docs = [
    "To reverse a string in Python, use s[::-1]. This is a common interview trick.",
    "The capital of France is Paris, and it is known for the Eiffel Tower.",
    "Write a function that returns the sum of two numbers. def add(a, b): return a + b",
]

eval_items = [
    "Write a function that returns the sum of two numbers.",
    "What is the capital of France?",
]


def word_ngrams(text, n=6):
    """Split text into consecutive n-word fragments for coarse overlap detection"""
    words = re.findall(r"[a-zA-Z]+", text.lower())
    return set(tuple(words[i:i + n]) for i in range(len(words) - n + 1))


def max_overlap(eval_text, train_texts):
    """Return the max n-gram overlap ratio between an eval question and the training set"""
    eval_grams = word_ngrams(eval_text, n=4)
    best = 0
    best_doc = None
    for doc in train_texts:
        train_grams = word_ngrams(doc, n=4)
        if not eval_grams:
            continue
        overlap = len(eval_grams & train_grams) / len(eval_grams)
        if overlap > best:
            best = overlap
            best_doc = doc
    return best, best_doc


for item in eval_items:
    score, doc = max_overlap(item, train_docs)
    status = "suspected contamination" if score >= 0.6 else "not hit"
    print(f"Eval question: {item}")
    print(f"Max overlap: {score:.0%} -> {status}")
    if doc:
        print(f"Hit document: {doc[:80]}...")
    print()

print("Key observation: decontamination checks overlap between the training set and the benchmark.")


### 2.7 Data-Juicer: Expressing a Pipeline as Configuration

**Data-Juicer** packages cleaning, filtering, deduplication, and sampling as composable operators controlled by YAML. Input and output commonly use JSONL:

```json
{"text":"document body", "meta":{"src":"common_crawl","date":"2024-06-01"}}
```

Operators preserve audit metadata and fall into five groups:

| Category | Suffix | Purpose | Example |
|:---|:---|:---|:---|
| mapper | `_mapper` | transform/redact | HTML cleaning, email redaction |
| filter | `_filter` | keep/drop by score | language, Perplexity |
| deduplicator | `_deduplicator` | remove repeats | document MinHash |
| selector | `_selector` | sample/rank | top-k by quality |
| grouper | `_grouper` | group statistics | length/source analysis |

A configuration makes the recipe reproducible and lets every decision be traced through metadata.


In [ ]:
# === Simulate Data-Juicer data format and an operator chain ===
# Use plain Python to simulate the chain without installing Data-Juicer
# In real use, install py-data-juicer and replace each function with its corresponding operator

import json

# Standard Data-Juicer input is jsonl with one document per line
raw_data = [
    {"text": "An introduction to <p>machine learning</p>; contact test@example.com",
     "meta": {"src": "web", "lang_score": 0.95}},
    {"text": "buy cheap pills online online online online online",
     "meta": {"src": "web", "lang_score": 0.10}},
    {"text": "Machine learning is a branch of AI that learns patterns from data.",
     "meta": {"src": "wiki", "lang_score": 0.99}},
]

def op_language_filter(doc, lang='zh', min_score=0.8):
    """Simulate language_id_score_filter: discard documents below the score threshold."""
    return doc if doc['meta']['lang_score'] >= min_score else None

def op_clean_html(doc):
    """Simulate clean_html_mapper by removing HTML tags."""
    import re
    doc['text'] = re.sub(r'<[^>]+>', '', doc['text'])
    return doc

def op_clean_email(doc):
    """Simulate clean_email_mapper by redacting email addresses."""
    import re
    doc['text'] = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '<EMAIL>', doc['text'])
    return doc

# Execute operators in the same order as the process list in a YAML configuration
pipeline = [op_language_filter, op_clean_html, op_clean_email]

print("=== Simulated Data-Juicer operator chain from a YAML process list ===")
print()
kept = []
for doc in raw_data:
    for op in pipeline:
        doc = op(doc) if doc else None
        if doc is None:
            break
    if doc:
        kept.append(doc)

print(f"Input {len(raw_data)} documents; retained {len(kept)} after filtering")
print()
for doc in kept:
    print(json.dumps(doc, ensure_ascii=False))
print()
print("Key observation: document 2 is removed by language_id_score_filter because its score is 0.10;")
print("document 1 has HTML and email removed by mapper operators, one operator per pipeline step")


#### Related Tools and Public Case Studies

Llama 3 demonstrates gains from 15T+ carefully processed Tokens; FineWeb publishes filtering and deduplication ablations over massive Common Crawl data; DCLM compares recipes under fixed training; Dolma documents an open 3T-Token pipeline; and RedPajama exposes quality and deduplication signals.

Data-Juicer provides a broad operator system, DataTrove emphasizes distributed large-scale text processing, NeMo Curator integrates GPU-accelerated curation, and specialized tools may focus on deduplication or PII. Select by pipeline scale, available compute, formats, and audit needs rather than brand name.


#### Production Filtering Recipes

Pretraining pipelines commonly combine exact deduplication, MinHash/LSH near-deduplication, optional semantic deduplication, heuristic statistics, KenLM Perplexity bands, learned quality classifiers, and an explicit source mixture. Llama 3, FineWeb/FineWeb-Edu, DCLM, Dolma, and CCNet publish representative variants.

Classifier filtering can bias toward encyclopedic and academic prose: stronger filters may improve academic benchmarks but hurt ordinary web tasks. Every threshold therefore needs downstream ablation.

SFT places even more weight on each example. AlpaGasus showed that a high-quality subset of Alpaca could outperform the full set; other work ranks instruction difficulty or response quality. Useful checks include correctness, instruction–answer consistency, completeness, diversity, style balance, refusal behavior, and contamination. There is no universal recipe: define a target capability, curate, train a proxy model, evaluate, and iterate.


### 2.8 A Data Funnel for a 1B Model

Extraction, quality filtering, safety, deduplication, and decontamination form one pipeline rather than independent jobs. We now simulate raw WARC → extraction → filtering → deduplication → mixture, printing counts at every stage. Production implementations are larger, but the funnel statistics—how much entered, why it was removed, and what remained—are the essential audit trail.


In [ ]:
print("=== Practice: 1B LLM Data Pipeline ===")
print()

steps = [
    ("Step 1: Determine data budget", [
        "Chinchilla optimal: N = 1B -> D ~ 20B tokens",
        "Over-training: N = 1B -> D ~ 100B tokens",
        "Choice: 50B tokens (a middle ground, good cost-efficiency)",
    ]),
    ("Step 2: Download Common Crawl", [
        "Download the most recent 2-3 month dumps (about 20TB compressed WARC)",
        "Tools: cc_downloader, HuggingFace datasets",
    ]),
    ("Step 3: Text extraction + language filtering", [
        "WARC -> HTML -> plain text (trafilatura / resiliparse)",
        "Language detection (fastText): keep only English and Chinese",
        "Output: ~2TB plain text (about 400B tokens)",
    ]),
    ("Step 4: Quality filtering", [
        "Heuristic rules: length/word-length/special-chars/line-repetition",
        "KenLM PPL filter: 10 < PPL < 1000",
        "Output: ~200GB (about 40B tokens) -> only 10% remains",
    ]),
    ("Step 5: Deduplication", [
        "Exact dedup: SHA256 hash -> removes ~10%",
        "MinHash approximate dedup: similarity > 80% keep only one -> removes ~20%",
        "Output: ~140GB (about 28B tokens)",
    ]),
    ("Step 6: Mix other sources", [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        "Others: 2B tokens",
        "Total: 28B + 22B = 50B tokens",
    ]),
    ("Step 7: Tokenize + packing", [
        "Use a BPE tokenizer to turn text into token IDs",
        "Concatenate into a continuous sequence, cut into 2048/4096 length chunks",
        "Insert <EOS> token at document boundaries",
        "Shuffle + pack into training batches -> start training!",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print("Compression ratio summary:")
print("  20TB WARC -> 2TB plain text -> 200GB filtered -> 140GB after dedup")
print("  Final usable data is only ~0.7% of the original download")

In [ ]:
# === Visualize the data funnel: how much remains after every stage? ===
# Values follow the preceding pipeline, converting 1 TB to 1,000 GB
import matplotlib.pyplot as plt

stages = ['Raw WARC', 'Extracted text', 'Quality filtered', 'Deduplicated']
sizes_gb = [20000, 2000, 200, 140]

fig, ax = plt.subplots(figsize=(7, 4.4))
bars = ax.barh(np.arange(len(stages)), sizes_gb, 0.55,
               color=['#94a3b8', '#60a5fa', '#2563eb', '#1e40af'])
ax.set_xscale('log')
ax.set_xlim(70, 90000)
ax.set_yticks(np.arange(len(stages)))
ax.set_yticklabels(stages)
ax.invert_yaxis()
ax.set_xlabel('Data size (GB, log scale)')
ax.set_title('The data funnel: 0.7% of the raw crawl survives')
for bar, size in zip(bars, sizes_gb):
    ratio = size / sizes_gb[0]
    if ratio >= 0.01:
        label = f'{size:,} GB ({ratio:.0%} of raw)'
    else:
        label = f'{size:,} GB ({ratio:.1%} of raw)'
    ax.text(size * 1.15, bar.get_y() + bar.get_height() / 2, label,
            va='center', fontsize=9)
plt.show()

print("Key observation: data engineering mostly discards data rather than retaining it.")
print("  Twenty TB of raw web pages becomes 140 GB; each stage removes another large fraction")
print("  Calibrate thresholds with small-model trials and evaluations: too strict hurts diversity, too loose hurts quality")


## 3. Building Your Own Data: From Recipes to Synthesis

Filtering answers whether data is clean. The next questions are what to train on, in what proportions, and how to fill missing domains. We create a mixture, validate it with benchmarks, add domain-specific sources, construct SFT examples, and synthesize additional data.

### 3.1 Data Mixtures: Repeat High-Quality Sources

Raw volume would let mediocre web text dominate small high-quality sources such as Wikipedia and papers. Sampling weights can repeat high-quality data for more epochs while showing lower-quality data fewer times. The next cell turns this idea into measurable proportions.


In [ ]:
# === Data-mixture strategy ===
print("=== Data Distillation Pipeline ===")
print()

sources = [
    ("Common Crawl, filtered", 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print(f"{'Source':<25s} {'Raw size':>10s} {'Quality':>6s} {'Epoch':>6s} {'Effective':>10s} {'Share':>8s}")
print("-" * 72)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<25s} {size:>6.0f}B   {quality:>5.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print(f"Total effective data: {total_effective:.0f}B tokens")
print()
print("Key observations:")
print("  Wikipedia has only 100B tokens, but four epochs produce 400B effective tokens")
print("  Common Crawl has 10T tokens, but one epoch avoids repeatedly learning noisy text")
print("  ArXiv is small but high quality, so four epochs amplify scientific reasoning data")
print()
print("Multiple epochs do not mean copying each article four times")
print("Instead, reshuffle between epochs so the model encounters a different order each time")


In [ ]:
# === Visualize raw volume versus effective training volume ===
import matplotlib.pyplot as plt

# Source names above may be localized, so use English labels in the chart in the same order
en_names = ['Common Crawl', 'Wikipedia', 'Books', 'Code (GitHub)',
            'ArXiv Papers', 'News']
raw_tokens = [size for _, size, _, _ in sources]
eff_tokens = [size * epochs for _, size, _, epochs in sources]

y = np.arange(len(sources))
height = 0.38

fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.barh(y - height/2, raw_tokens, height, color='#94a3b8', label='Raw tokens')
ax.barh(y + height/2, eff_tokens, height, color='#2563eb',
        label='Effective tokens = raw x epochs')
ax.set_yticks(y)
ax.set_yticklabels(en_names)
ax.invert_yaxis()
ax.set_xlabel('Tokens (B)')
ax.set_title('Epochs amplify high-quality sources in the mix')
for i, (r, e) in enumerate(zip(raw_tokens, eff_tokens)):
    ax.text(r + 90, i - height/2, f'{r:.0f}B', va='center', fontsize=8)
    ax.text(e + 90, i + height/2, f'{e:.0f}B', va='center', fontsize=8)
ax.set_xlim(0, 12500)
ax.legend(loc='lower right', fontsize=8)
plt.show()

print("Key observation: Wikipedia has only 1% of Common Crawl's raw volume,")
print("  but four epochs quadruple its effective amount, giving it far more loss weight than its raw share")
print("  Mixture design asks what the model should learn more often, not merely what data is available")


### 3.2 The Data Recipe

A **data recipe** is an executable specification of filtering, sampling, and mixing. It is iterative:

```text
Recipe A: more web, less code
Recipe B: more code and math
Recipe C: only high-scoring web, fewer total Tokens
```

Train small proxy models and compare loss plus MMLU, GSM8K, and HumanEval. If code improves while general knowledge falls, the mixture has traded too much broad text for code. Data-centric LLM development tunes recipes as deliberately as architecture.


In [ ]:
# === Data Recipe A/B comparison demo ===
print("=== Data Recipe: Recipe Comparison ===")
print()

recipes = {
    "A_web_heavy": {
        "web": 0.70,
        "wiki": 0.10,
        "books": 0.10,
        "code": 0.05,
        "math": 0.05,
    },
    "B_code_math": {
        "web": 0.45,
        "wiki": 0.10,
        "books": 0.10,
        "code": 0.25,
        "math": 0.10,
    },
    "C_quality_web": {
        "web": 0.50,
        "wiki": 0.20,
        "books": 0.15,
        "code": 0.10,
        "math": 0.05,
    },
}

# These scores are teaching simulations: showing how to compare recipes, not real model results.
eval_scores = {
    "A_web_heavy": {"MMLU": 54, "GSM8K": 28, "HumanEval": 12},
    "B_code_math": {"MMLU": 52, "GSM8K": 39, "HumanEval": 24},
    "C_quality_web": {"MMLU": 57, "GSM8K": 31, "HumanEval": 16},
}

header = (
    f"{'Recipe':<15s} {'web':>5s} {'wiki':>5s} {'books':>6s} "
    f"{'code':>6s} {'math':>6s}  {'MMLU':>6s} {'GSM8K':>6s} {'HEval':>6s}"
)
print(header)
print("-" * len(header))
for name, mix in recipes.items():
    scores = eval_scores[name]
    print(
        f"{name:<15s} "
        f"{mix['web']:>4.0%} {mix['wiki']:>5.0%} {mix['books']:>6.0%} "
        f"{mix['code']:>6.0%} {mix['math']:>6.0%}  "
        f"{scores['MMLU']:>6.1f} {scores['GSM8K']:>6.1f} {scores['HumanEval']:>6.1f}"
    )

print()
print("Key observation: no recipe is absolutely best; it depends on whether you care more about general ability, math, or code.")


In [ ]:
# === Visualize the data mixtures of three recipes ===
import matplotlib.pyplot as plt

sources = ["web", "wiki", "books", "code", "math"]
recipe_names = list(recipes.keys())

fig, ax = plt.subplots(figsize=(8, 3.5))
left = [0.0] * len(recipe_names)
for src in sources:
    vals = [recipes[r][src] for r in recipe_names]
    ax.barh(recipe_names, vals, left=left, label=src)
    left = [l + v for l, v in zip(left, vals)]
ax.set_xlim(0, 1)
ax.set_xlabel("Mix fraction")
ax.set_title("Data mixing ratios of three recipes")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.3))
plt.tight_layout()
plt.show()


### 3.3 Validating a Recipe with Benchmarks

Benchmarks are evaluation rulers, not normally pretraining sources. They serve two roles: measure whether a recipe improves intended abilities and check contamination.

| Ability | Benchmarks | Approximate Focus |
|:---|:---|:---|
| General knowledge | MMLU, MMLU-Pro, C-Eval, CMMLU, AGIEval | multi-subject exams |
| Mathematics | GSM8K, MATH, AIME | multi-step quantitative reasoning |
| Code | HumanEval(+), MBPP, LiveCodeBench | executable solutions |
| Complex reasoning | BBH, GPQA, ARC-Challenge | difficult abstract/scientific tasks |
| Common sense | HellaSwag, PIQA, WinoGrande | completion and physical/social reasoning |

Use multiple benchmarks and preserve a held-out business evaluation; one public score cannot characterize a recipe.


In [ ]:
# === Benchmark profile: identify which abilities changed after a data-recipe change ===
print("=== Benchmark ability profile: Recipe A versus Recipe B ===")
print()

benchmarks = [
    ("MMLU", "general"),
    ("C-Eval", "Chinese"),
    ("GSM8K", "word math"),
    ("MATH", "contest math"),
    ("HumanEval", "code funcs"),
    ("MBPP", "code tasks"),
    ("BBH", "reasoning"),
    ("IFEval", "instruction"),
]

# Simulated teaching scores illustrate how to read the table; they are not real model results
recipe_a = {
    "MMLU": 54,
    "C-Eval": 50,
    "GSM8K": 28,
    "MATH": 12,
    "HumanEval": 12,
    "MBPP": 18,
    "BBH": 36,
    "IFEval": 52,
}
recipe_b = {
    "MMLU": 52,
    "C-Eval": 48,
    "GSM8K": 39,
    "MATH": 20,
    "HumanEval": 24,
    "MBPP": 31,
    "BBH": 39,
    "IFEval": 51,
}

print(f"{'Benchmark':<12s} {'Ability':<8s} {'A':>5s} {'B':>5s} {'Change':>7s}  Observation")
print("-" * 62)
for name, ability in benchmarks:
    delta = recipe_b[name] - recipe_a[name]
    if delta >= 5:
        note = "Both pass through"
    elif delta <= -3:
        note = "Both pass through"
    else:
        note = "Both pass through"
    print(
        f"{name:<12s} {ability:<8s} "
        f"{recipe_a[name]:>5.1f} {recipe_b[name]:>5.1f} {delta:>+7.1f}  {note}"
    )

print()
print("Key observation: Recipe B resembles adding code and math data; those scores rise while general knowledge dips.")
print("That is not inherently better or worse; it depends on whether the target is general or specialized.")


In [ ]:
# === Visualize the ability profiles of Recipes A and B ===
import matplotlib.pyplot as plt

bench_names = [n for n, _ in benchmarks]
scores_a = [recipe_a[n] for n in bench_names]
scores_b = [recipe_b[n] for n in bench_names]

y = np.arange(len(bench_names))
height = 0.38

fig, ax = plt.subplots(figsize=(7.5, 4.8))
ax.barh(y - height/2, scores_a, height, color='#94a3b8',
        label='Recipe A (web-heavy)')
ax.barh(y + height/2, scores_b, height, color='#2563eb',
        label='Recipe B (code + math)')
ax.set_yticks(y)
ax.set_yticklabels(bench_names)
ax.invert_yaxis()
ax.set_xlabel('Score')
ax.set_xlim(0, 72)
ax.set_title('Same data budget, different ability profile')
for i, (a, b) in enumerate(zip(scores_a, scores_b)):
    delta = b - a
    if delta >= 5:
        color = '#16a34a'
    elif delta <= -3:
        color = '#dc2626'
    else:
        color = '#64748b'
    ax.text(max(a, b) + 1.5, i, f'{delta:+d}', va='center',
            fontsize=9, color=color)
ax.legend(loc='lower right', fontsize=8)
plt.show()

print("Key observation: read the profile, not a single total score.")
print("  Positive green values on GSM8K, MATH, HumanEval, and MBPP show the effect of more code and math")
print("  Negative red values on MMLU and C-Eval show that some general text was displaced")
print("  The next recipe change depends on whether the goal is a general or specialized model")


#### Benchmark Tools

`lm-evaluation-harness` is common for reproducible open-model benchmarks. EvalScope combines capability and performance evaluation with ModelScope integrations. OpenCompass supports configuration-driven, distributed multi-dataset reports. OpenAI Evals supports custom product evaluations, while HELM emphasizes broad scenario coverage and transparent reporting. Whichever tool you choose, pin prompts, templates, model revision, few-shot settings, and metrics.


### 3.4 Adding Domain Data: Mathematics, Code, Chinese, and More

For mathematics, pretraining sources include OpenWebMath, MathPile, Proof-Pile-2, and DeepSeekMath Corpus; SFT sources include MetaMathQA and NuminaMath-CoT; GSM8K and MATH are primarily evaluation sets and require decontamination.

For code, The Stack v2, StarCoderData, permissively licensed repositories, documentation, tests, and issue/patch pairs support pretraining and post-training. Preserve license metadata and remove secrets.

For Chinese, combine filtered Chinese web, encyclopedic/news/books, code and STEM text, plus instruction mixtures such as COIG and Infinity-Instruct. Language identification, script balance, tokenizer fertility, and translated-data artifacts need separate measurement.

The pattern generalizes: define the missing capability, locate PT/SFT/evaluation sources, audit licenses and leakage, choose mixture weight, and verify with targeted held-out tasks.


### 3.5 Building a Custom Dataset

```text
1. Define the missing capability with ten gold examples.
2. Collect licensed public data, internal documents, or crawled pages.
3. Clean, redact, deduplicate, and filter language.
4. Construct QA pairs by templates, document extraction, or synthesis.
5. Audit correctness, format, truncation, repetition, and diversity.
6. Run a small training experiment and targeted evaluation.
7. Convert discovered failure patterns into new filters and repeat.
```

Templates fit structured FAQ/table/API data; document-to-QA fits domain corpora; model synthesis expands a small seed set. Manual sampling is indispensable because automated scores cannot reveal every systematic error.


In [ ]:
# === Practice: turn three raw passages into a tiny SFT dataset ===
# Run Steps 3-5 end to end: cleaning, QA extraction, and quality inspection
import json, re

# Step 2 input: three passages from an internal wiki, deliberately containing noise
raw_docs = [
    "Model evaluation metrics include accuracy, precision, and recall. Accuracy is the<br>fraction of correct predictions.",
    "Ways to reduce overfitting include adding data, regularization, and early stopping.%%%%junk ad%%%%",
    "Gradient descent is an optimization algorithm that updates parameters in the direction of fastest loss decrease.",
]

# Step 3: clean with the equivalent of clean_html_mapper plus a custom rule
def clean(text):
    text = re.sub(r'<[^>]+>', '', text)      # Remove HTML tags
    text = re.sub(r'%{2,}[^%]*%{2,}', '', text)  # Remove advertisement fragments
    return re.sub(r'\s+', ' ', text).strip()

cleaned = [clean(d) for d in raw_docs]

# Step 4: extract QA pairs by rewriting concept definitions as instructions
# This teaching example uses rules; a real pipeline can use rules or an LLM
qa_pairs = []
concepts = {
    "accuracy": "What is accuracy?",
    "overfitting": "What are ways to reduce overfitting?",
    "gradient descent": "What is gradient descent?",
}
for text in cleaned:
    for concept, question in concepts.items():
        if concept in text:
            qa_pairs.append({"instruction": question, "input": "", "output": text})
            break  # Produce one question per document to avoid duplicates

# Step 5: inspect quality with rules plus a human-review checklist
def quality_check(sample):
    """Return a list of problems; an empty list means pass."""
    problems = []
    if len(sample["output"]) < 20:
        problems.append("answer too short")
    if "http" in sample["output"].lower():
        problems.append("link remains")
    if sample["output"].endswith(("。", "！", "？")) is False:
        problems.append("answer may be truncated")
    return problems

print("=== Data Distillation Pipeline ===")
print()
for s in qa_pairs:
    problems = quality_check(s)
    status = "pass" if not problems else "problems: " + ", ".join(problems)
    print(f"Q: {s['instruction']}")
    print(f"A: {s['output'][:50]}...")
    print(f"Quality check: {status}")
    print()

# Save in Alpaca format, which training frameworks can read directly
# with open('my_sft_dataset.jsonl', 'w') as f:
#     for s in qa_pairs:
#         f.write(json.dumps(s, ensure_ascii=False) + '\n')
print(f"Key observation: three passages produced {len(qa_pairs)} examples. The dataset is tiny, but its format,")
print("cleaning, and inspection pipeline is the same at million-example scale: validate small before scaling up.")


In [ ]:
# === Visualize the quality profile of the custom dataset ===
import matplotlib.pyplot as plt

# Two common checks: response-length distribution and duplication
lengths = [len(s['output']) for s in qa_pairs]
dup_ratio = len(lengths) / len(set(s['output'] for s in qa_pairs))  # Ratio before and after deduplication

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].bar(range(1, len(lengths) + 1), lengths, color='#4C72B0')
axes[0].axhline(20, color='red', linestyle='--', label='min length threshold')
axes[0].set_xlabel('Sample index')
axes[0].set_ylabel('Output length (chars)')
axes[0].set_title('Output length per sample')
axes[0].legend()

axes[1].bar(['samples', 'unique outputs'],
            [len(lengths), len(set(s['output'] for s in qa_pairs))],
            color=['#4C72B0', '#DD8452'])
axes[1].set_ylabel('Count')
axes[1].set_title(f'Duplication check (ratio = {dup_ratio:.2f})')

plt.tight_layout()
plt.show()
print("Key observation: the left chart checks each response against the minimum length; the right compares")
print("sample and unique-output counts. A large gap indicates template repetition and insufficient diversity.")


### 3.6 Synthetic Data

Post-training requires structured instruction/response and preference records, but expert annotation is expensive and too slow for frequent iteration. Synthetic pipelines use a strong permitted teacher model to generate instructions, responses, critiques, preferences, or reasoning traces, then filter them before training a Target.

Major patterns include Self-Instruct from seed tasks, Evol-Instruct that increases complexity, teacher distillation, and STaR that retains verified self-generated reasoning. Synthesis reduces marginal cost but transfers teacher biases and demands legal, policy, quality, and contamination checks.


#### Self-Instruct

Stanford's 2022 Self-Instruct paper first systematically proposed: treat the model as an instruction generator.

The process has four steps:

1. **Seed pool**: humans write 100-200 instructions as seeds (covering different task types)
2. **Generation**: sample a few from the seed pool and let the model generate new instructions
3. **Response**: let the model write responses to its own new instructions
4. **Filtering**: use rules (dedup, low-quality filtering) + model scoring, keep high-quality samples and add them to the seed pool

Repeat steps 2-4, and the seed pool grows exponentially. The original paper expanded from 175 seeds to 52K instructions, costing less than $500.

Below we simulate this process in Python, making the "generate-filter" loop explicit.

In [ ]:
# === Self-Instruct process simulation ===
# Demo: from 3 seed instructions, run two rounds of self-generation, observe seed pool growth

import random
random.seed(42)

# Step 1: human seed instructions (100-200 in real scenarios)
seed_pool = [
    "Translate this sentence into English: the weather is nice today",
    "Summarize the main points of the following passage: ...",
    "Explain what gravity is to a 5-year-old",
]

# Step 2: define the simulation of "model generation"
# In real scenarios call GPT-4 / Claude; here we simulate with preset results
def mock_generate_instructions(seeds, n=3):
    """Simulate: model sees seeds, generates new instructions"""
    templates = [
        "Rewrite this passage in a {style} style: ...",
        "Rewrite the following in {lang}: ...",
        "Explain {concept} to an {age}-year-old: ...",
        "Summarize this article about {topic}: ...",
        "Compare the similarities and differences of {a} and {b}: ...",
    ]
    fills = [
        {"style": "formal"}, {"style": "colloquial"}, {"lang": "French"},
        {"age": "8"}, {"age": "high schooler"}, {"concept": "photosynthesis"},
        {"concept": "quantum mechanics"}, {"topic": "climate change"}, {"a": "Python", "b": "Go"},
    ]
    new = []
    for _ in range(n):
        t = random.choice(templates)
        f = random.choice(fills)
        try:
            new.append(t.format(**f))
        except KeyError:
            new.append(t)
    return new

# Step 3: filtering (dedup + length + keywords)
def filter_instructions(candidates, existing):
    kept = []
    seen = set(existing)
    for inst in candidates:
        if inst in seen:
            continue
        if len(inst) < 5:
            continue
        kept.append(inst)
        seen.add(inst)
    return kept

# Run two rounds
for round_id in range(1, 3):
    seeds_sample = random.sample(seed_pool, k=min(3, len(seed_pool)))
    new_insts = mock_generate_instructions(seeds_sample, n=5)
    kept = filter_instructions(new_insts, seed_pool)
    seed_pool.extend(kept)
    print(f"=== Round {round_id} ===")
    print(f"  Generated this round: {len(new_insts)} items")
    print(f"  Kept after filtering: {len(kept)} items")
    print(f"  Seed pool total: {len(seed_pool)} items")
    for inst in kept:
        print(f"    + {inst}")
    print()

print("=" * 60)
print("Key observations:")
print(f"  Seed pool went from 3 -> {len(seed_pool)} items (two rounds)")
print(f"  Real Self-Instruct runs 10+ rounds, from 175 -> 52K items")
print(f"  Filtering determines quality: dedup avoids mode collapse into a few templates")

#### Evol-Instruct

Self-Instruct has a problem: the new instructions are "too similar" to the seeds, and the task difficulty distribution is narrow. The WizardLM paper proposed Evol-Instruct -- instead of generating from scratch, it **evolves** existing instructions.

Three directions of evolution:

- **In-depth evolution**: make the instruction more complex. Example: `compute 2+2` -> `compute (3.14 x 7 + 2) / 0.5 and explain each step`
- **In-breadth evolution**: change the topic but keep the difficulty. Example: `write a Python sort function` -> `write a Python dedup function`
- **Eliminating**: filter out failed generated instructions

This systematically covers the full "easy-medium-hard" difficulty distribution.

In [ ]:
# === Evol-Instruct demo ===
# Run 3 rounds of in-depth evolution on the same seed instruction

seed_instruction = "Write a function that takes two numbers and returns the larger one"

# Prompt templates simulating in-depth evolution (fed to GPT-4 in real scenarios)
evolution_prompts = [
    """Make the following instruction more complex, adding 1-2 constraints.
    Instruction: {inst}
    New instruction:""",
    """Rewrite the following instruction into a more professional version, requiring boundary condition handling.
    Instruction: {inst}
    New instruction:""",
    """Extend the following instruction into a multi-step task, adding exception handling requirements.
    Instruction: {inst}
    New instruction:""",
]

# Simulated evolution results (LLM output in real scenarios)
evolved_chain = [
    seed_instruction,
    "Write a function that takes two floats and returns the larger one; require handling of NaN",
    "Write a Python function max_of_two(a, b) that handles floats, NaN, Inf, "
    "with type annotations and docstring; require passing mypy static checks",
    "Implement max_of_two(a, b), supporting mixed comparison of int/float/Decimal types; "
    "raise a custom TypeError; write a complete pytest test suite covering boundary conditions",
]

print("=== Evol-Instruct In-Depth Evolution Chain ===\n")
for i, inst in enumerate(evolved_chain):
    if i == 0:
        print(f"[seed]   {inst}")
    else:
        print(f"[gen {i}] {inst}")
    print()

print("=" * 60)
print("Key observations:")
print("  In-depth evolution drives the instruction deeper on the 'same topic'")
print("  After 3 generations, it went from a 1-line code task to a 30-line engineering task")
print("  WizardLM used this method to evolve 250 seeds into 250K high-quality instructions")

#### Distilling Data from a Strong Teacher

1. Prepare prompts from permitted logs, open instruction sets, or Self-Instruct.
2. Generate responses with a strong teacher that permits this use.
3. Filter for correctness, safety, diversity, and style.
4. Train the smaller model with the resulting pairs.

Alpaca popularized inexpensive API-generated instruction data. In production, contract and Terms-of-Service compliance are essential; organizations often use an internally owned teacher.


In [ ]:
# === Distillation data generation template ===
# Demo: call an OpenAI-compatible API to batch-turn a prompt list into training data

# Real run requires: pip install openai and setting an API key
# Below only shows the data flow, no actual call

def distill_from_strong_model(prompts, model_name="qwen-plus",
                              api_key="sk-xxx", base_url=None):
    """
    Use a strong model to generate responses for a batch of prompts, return SFT training pairs.
    Real scenarios add: concurrency, retries, length filtering, diversity filtering.
    """
    from openai import OpenAI
    client = OpenAI(api_key=api_key, base_url=base_url)

    distilled = []
    for prompt in prompts:
        # Real scenarios wrap a system prompt with a chat template
        resp = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ],
            temperature=0.7,  # diversity
            max_tokens=1024,
        )
        answer = resp.choices[0].message.content
        distilled.append({"prompt": prompt, "response": answer})
    return distilled


# Simulated data: 3 prompts, showing the training-pair format
demo_prompts = [
    "Implement an LRU cache in Python",
    "Explain the principle of batch normalization",
    "Translate 'I'd like to book a ticket to Beijing' into English",
]

# Simulated strong-model output (returned by the API in real scenarios)
mock_distilled = [
    {"prompt": demo_prompts[0],
     "response": "from collections import OrderedDict\nclass LRUCache: ..."},
    {"prompt": demo_prompts[1],
     "response": "Batch Norm normalizes each layer's input along the batch dimension..."},
    {"prompt": demo_prompts[2],
     "response": "I'd like to book a ticket to Beijing."},
]

print("=== Distilled SFT Training Pairs ===\n")
for i, pair in enumerate(mock_distilled, 1):
    print(f"--- Training pair #{i} ---")
    print(f"prompt:   {pair['prompt']}")
    print(f"response: {pair['response'][:80]}...")
    print()

print("=" * 60)
print("Key observations:")
print("  Distillation cost ~= API call cost (Alpaca 52K items ~= $600)")
print("  The bottleneck in real scenarios is not money, it is prompt pool diversity")
print("  Industrial practice: mix real user logs + Self-Instruct + Evol")

#### STaR

STaR (Self-Taught Reasoner) lets a model generate reasoning for questions with verifiable answers. Correct trajectories enter the training set; incorrect attempts may receive the answer as a hint and try a rationalization. Retrain on the expanded set and repeat. Correct self-generated traces are well matched to the model's current capability while pointing toward valid solutions.


In [ ]:
# === STaR process simulation ===
# Demo: run one round of STaR, observe how the training set expands

import random
random.seed(42)

# Prepare a question bank (with standard answers)
problems = [
    {"q": "3 x 7 + 2 = ?", "a": "23"},
    {"q": "If 2x = 10, x = ?", "a": "5"},
    {"q": "sqrt(81) = ?", "a": "9"},
    {"q": "How many times is 12 of 3?", "a": "4"},
    {"q": "(5 + 3) x 2 = ?", "a": "16"},
]

# Simulate the current model's "reasoning + answering" ability (accuracy about 60%)
def mock_model_generate(problem):
    q = problem["q"]
    correct_a = problem["a"]
    # 60% chance of correct answer
    if random.random() < 0.6:
        return {"reasoning": f"derive {q} -> got {correct_a}", "answer": correct_a, "correct": True}
    else:
        wrong = str(int(correct_a) + random.choice([-2, -1, 1, 2]))
        return {"reasoning": f"derive {q} -> miscalculated -> {wrong}", "answer": wrong, "correct": False}

# Run one round of STaR
training_set = []
print("=== STaR One Iteration ===\n")
for p in problems:
    out = mock_model_generate(p)
    status = "include" if out["correct"] else "drop"
    print(f"Question: {p['q']}")
    print(f"  Reasoning: {out['reasoning']}")
    print(f"  Answer: {out['answer']}  Correct answer: {p['a']}  {status}")
    if out["correct"]:
        training_set.append({
            "prompt": p["q"],
            "reasoning": out["reasoning"],
            "answer": out["answer"],
        })

print(f"\n{'=' * 60}")
print(f"Round result:")
print(f"  Total questions: {len(problems)}")
print(f"  Training pairs included: {len(training_set)} ({len(training_set)/len(problems)*100:.0f}%)")
print(f"\nKey observations:")
print(f"  STaR solidifies the reasoning process the model 'can answer correctly' into training data")
print(f"  Wrong samples are dropped, not polluting the training set")
print(f"  After multiple iterations, the model's reasoning ability keeps improving (math accuracy 30% -> 50%+ in the paper)")

### 3.7 The Data Flywheel and Its Limits

```text
train model v_n → generate/evolve/distill/verify examples
→ filter + deduplicate + score → train v_{n+1} → repeat
```

Risks include **mode collapse** toward already-easy tasks, **error amplification** of systematic mistakes, **distribution drift** away from real users, benchmark leakage, and synthetic style homogenization. Countermeasures include real-data anchors, explicit difficulty expansion, independent verifiers, human audits, source labels, and held-out evaluations. A flywheel improves only when its quality gate is stronger than its error feedback loop.


## 4. From Documents to the Training Stream

Tokenize documents, insert `<EOS>`, concatenate them into one Token stream, and cut fixed-length chunks. Chunk boundaries may cross documents, so `<EOS>` tells the model that a new document begins. Track source and document boundaries even if the final tensor is contiguous; they are useful for masking and audit.


### 4.1 Sequence Packing

Padding short documents wastes computation. **Sequence Packing** uses a bin-packing strategy to place multiple examples into one fixed-length sequence, often raising utilization dramatically.

Documents inside a packed sequence must not attend across boundaries when the task assumes independence. A block-diagonal Attention mask and per-document position handling prevent leakage. Packing improves throughput, not the amount of training signal, so report both raw Tokens and effective loss Tokens.


In [ ]:
# === Sequence Packing hand-calculation demo ===
print("=== Sequence Packing: Bin-Packing Algorithm Hand Calculation ===")
print()

# Simulate the token lengths of 8 documents
docs = [
    ("Doc A", 120),
    ("Doc B", 80),
    ("Doc C", 200),
    ("Doc D", 50),
    ("Doc E", 150),
    ("Doc F", 90),
    ("Doc G", 60),
    ("Doc H", 40),
]

max_seq_len = 256  # fixed chunk length

# First-Fit Decreasing bin packing: sort long docs first, then pack into chunks one by one
sorted_docs = sorted(docs, key=lambda x: -x[1])  # sort by length descending
chunks = []  # each chunk is (doc list, used length)

for name, length in sorted_docs:
    placed = False
    # Try to fit into an existing chunk
    for chunk in chunks:
        if chunk[1] + length + 1 <= max_seq_len:  # +1 for the EOS token
            chunk[0].append((name, length))
            chunk[1] += length + 1
            placed = True
            break
    if not placed:
        chunks.append([[(name, length)], length + 1])

print(f"Total documents: {len(docs)}")
print(f"Chunk capacity: {max_seq_len} tokens")
print(f"Packing result: {len(chunks)} chunks")
print()

total_used = 0
for i, (items, used) in enumerate(chunks):
    doc_names = ' + '.join(name for name, _ in items)
    padding = max_seq_len - used
    util = used / max_seq_len * 100
    total_used += used
    print(f"  Chunk {i+1}: [{doc_names}]")
    print(f"          used {used} tokens, padding {padding}, utilization {util:.0f}%")

overall_util = total_used / (len(chunks) * max_seq_len) * 100
print(f"\nOverall utilization: {overall_util:.0f}%")
print()
print("Compared with the naive approach (one chunk per document):")
naive_chunks = len(docs)
print(f"  Naive: {naive_chunks} chunks")
print(f"  Packing: {len(chunks)} chunks")
print(f"  Saved: {(1 - len(chunks)/naive_chunks)*100:.0f}%")

In [ ]:
# === Visualize naive chunking versus Sequence Packing ===
import matplotlib.pyplot as plt

# Convert localized document names to English labels so all chart text is English
doc_en = {name: name.replace('Document', 'Doc ') for name, _ in docs}
palette = ['#2563eb', '#16a34a', '#f59e0b', '#dc2626',
           '#7c3aed', '#0891b2', '#db2777', '#65a30d']
color_of = {name: palette[i] for i, (name, _) in enumerate(docs)}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5, 5.4),
                               height_ratios=[len(docs), len(chunks)])

# Top: naive approach gives each document its own chunk and fills the remainder with padding
for i, (name, length) in enumerate(docs):
    ax1.broken_barh([(0, length)], (i - 0.4, 0.8), facecolors=color_of[name])
    ax1.broken_barh([(length, max_seq_len - length)], (i - 0.4, 0.8),
                    facecolors='#e2e8f0', hatch='///', edgecolor='white')
    ax1.text(length / 2, i, doc_en[name], ha='center', va='center',
             fontsize=8, color='white')
ax1.set_xlim(0, max_seq_len)
ax1.set_ylim(len(docs) - 0.5, -0.5)
ax1.set_title(f'Naive: 1 doc per chunk, {len(docs)} chunks')
ax1.set_ylabel('Chunk')

# Bottom: packing combines short documents in one chunk, leaving padding mainly at the end
for j, (items, used) in enumerate(chunks):
    start = 0
    for name, length in items:
        ax2.broken_barh([(start, length)], (j - 0.4, 0.8),
                        facecolors=color_of[name])
        ax2.text(start + length / 2, j, doc_en[name], ha='center',
                 va='center', fontsize=8, color='white')
        start += length + 1   # Add one EOS token after each document
    if used < max_seq_len:
        ax2.broken_barh([(start, max_seq_len - start)], (j - 0.4, 0.8),
                        facecolors='#e2e8f0', hatch='///', edgecolor='white')
ax2.set_xlim(0, max_seq_len)
ax2.set_ylim(len(chunks) - 0.5, -0.5)
ax2.set_title(f'FFD packing: {len(chunks)} chunks, utilization {overall_util:.0f}%')
ax2.set_xlabel('Position in chunk (tokens)')

plt.tight_layout()
plt.show()

print(f"Key observation: the same {len(docs)} documents need {len(docs)} naive chunks but only {len(chunks)} packed chunks.")
print("  Hatched cells are padding, which consumes most training compute under naive chunking")
print("  After packing, only the final chunk has substantial unused space")


### 4.2 Block-Diagonal Attention Mask

After packing, a chunk may contain 3-4 different documents. Without any restriction, the model "sees" the content of other documents during attention -- this is information leakage.

Solution: **block-diagonal mask**. Each document can only attend within itself; attention weights between different documents are set to negative infinity (becoming 0 after softmax).

```
Chunk: [Doc A][Doc B][Doc C]

Attention Mask:
        A A A B B C C
    A [ 1 1 1 0 0 0 0 ]   <- A only sees A
    A [ 1 1 1 0 0 0 0 ]
    A [ 1 1 1 0 0 0 0 ]
    B [ 0 0 0 1 1 0 0 ]   <- B only sees B
    B [ 0 0 0 1 1 0 0 ]
    C [ 0 0 0 0 0 1 1 ]   <- C only sees C
    C [ 0 0 0 0 0 1 1 ]

Blocks on the diagonal are each independent = block-diagonal
```

In [ ]:
import numpy as np

# === Block-Diagonal Mask construction demo ===
print("=== Block-Diagonal Attention Mask ===")
print()

# Assume a packed chunk contains 3 documents of lengths 3, 2, 2
doc_lengths = [3, 2, 2]
total_len = sum(doc_lengths)  # 7

# Build a block-diagonal mask (pure numpy)
mask = np.zeros((total_len, total_len))

pos = 0
for length in doc_lengths:
    for i in range(length):
        for j in range(i + 1):
            mask[pos + i, pos + j] = 1.0
    pos += length

print(f"Document lengths: {doc_lengths}, total length: {total_len}")
print()
print("Block-Diagonal Mask (1=visible, 0=masked):")
print("      ", "  ".join([f"t{i}" for i in range(total_len)]))
labels = []
pos = 0
for doc_idx, length in enumerate(doc_lengths):
    for _ in range(length):
        labels.append(f"D{doc_idx}")
print()
for i in range(total_len):
    row = "  ".join([f" {int(mask[i,j])}" for j in range(total_len)])
    print(f"  {labels[i]} t{i}: {row}")

print()
print("Observations:")
print("  D0 (Doc 0) internal: t0->t0, t1->t0,t1, t2->t0,t1,t2  <- causal attention")
print("  D1 (Doc 1) internal: t3->t3, t4->t3,t4               <- fully isolated from D0")
print("  D2 (Doc 2) internal: t5->t5, t6->t5,t6               <- isolated from other docs")
print()
print("In real training you also need:")
print("  1. Each document's position id restarts from 0")
print("  2. Loss is computed only on real tokens; padding does not count")


In [ ]:
# === Visualize a block-diagonal mask at a glance ===
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4.5))
plt.imshow(mask, cmap="Blues", vmin=0, vmax=1)
# Red lines mark document boundaries for lengths 3, 2, and 2: between t2/t3 and t4/t5
for boundary in (2.5, 4.5):
    plt.axhline(boundary, color="red", linewidth=1.2)
    plt.axvline(boundary, color="red", linewidth=1.2)
plt.xlabel("Key position (t)")
plt.ylabel("Query position (t)")
plt.title("Block-diagonal attention mask (3 docs)")
plt.colorbar(label="Visible (1) / masked (0)")
plt.tight_layout()
plt.show()


### 4.3 Fill-in-the-Middle

Code completion often has context on both sides of a cursor. Fill-in-the-Middle randomly divides code into prefix, middle, and suffix, then rearranges it:

```text
<PRE> prefix <SUF> suffix <MID> middle
```

The model observes prefix and suffix and predicts the missing middle. Special Tokens define the regions; depending on the objective, context labels may be ignored and loss applied primarily to the middle. FIM teaches infilling without changing the Decoder-only architecture.


In [ ]:
import numpy as np

# Hand-calculation demo of FIM format conversion

def fim_transform(code):
    """
    Convert a piece of code to the FIM training format.

    Randomly choose a cut point -> split into prefix/middle/suffix -> rearrange in FIM format
    """
    lines = code.split('\n')
    n = len(lines)

    # Randomly choose cut points
    prefix_len = np.random.randint(1, max(2, int(n * 0.4)))
    middle_len = np.random.randint(1, max(2, int(n * 0.5)))
    prefix_len = min(prefix_len, n - 1)
    middle_len = min(middle_len, n - prefix_len)

    prefix = '\n'.join(lines[:prefix_len])
    middle = '\n'.join(lines[prefix_len:prefix_len + middle_len])
    suffix = '\n'.join(lines[prefix_len + middle_len:])

    fim_text = f"<PRE> {prefix}\n<SUF> {suffix}\n<MID> {middle}"
    return fim_text, prefix, middle, suffix


np.random.seed(42)

sample_code = """def calculate(x, y):
    z = x + y
    result = z * 2
    if result > 100:
        return result
    else:
        return 0

print(calculate(3, 4))"""

fim_text, prefix, middle, suffix = fim_transform(sample_code)

print("=== FIM Format Conversion Hand Calculation ===")
print()
print("Original code:")
print(sample_code)
print()
print("--- After splitting ---")
print(f"Prefix ({len(prefix)} chars, model sees):")
print(prefix)
print()
print(f"Middle ({len(middle)} chars, training target):")
print(middle)
print()
print(f"Suffix ({len(suffix)} chars, model sees):")
print(suffix if suffix else '(empty)')
print()
print("--- FIM format (model input) ---")
print(fim_text)
print()

print("=== Label Mask During Training ===")
print()
print("Rules:")
print("  <PRE> ... <SUF> ... <MID> these special tokens -> known but not predicted")
print("  prefix and suffix content        -> label = ignore_index")
print("  middle content                   -> label = normal token (participates in loss)")
print()
print("Key observations:")
print("1. FIM rearranges text so the model can leverage bidirectional context during training")
print("2. <PRE> and <SUF> provide front/back context; after <MID> the model predicts the removed part")
print("3. Only the loss in the middle region is kept -- prefix/suffix give info but no penalty")
print("4. About 50% samples use standard format + 50% use FIM -> balance autoregressive generation and code completion")

## 5. Measuring Data Volume with Effective Tokens

File GB and row counts do not measure training signal. **Effective training Tokens** are the Tokens that actually contribute to loss after tokenization, truncation, masking, and packing. Pretraining usually supervises nearly every text Token; SFT often supervises only assistant responses even though prompts still consume compute. The next example estimates both for a small MiniMind dataset and compares them with published model scales.


In [ ]:
# === Estimate effective tokens for PT and SFT ===
# Core formulas:
#   PT effective tokens = sum(min(tokens per sample, max_seq_len))
#   SFT effective tokens = sum(assistant label tokens per sample), excluding the prompt

print("=== Input token IDs ===")
print()

# MiniMind data sizes from actual statistics
pt_mini = {"name": "pretrain_t2t_mini", "size_gib": 1.16, "lines": 1_270_238, "max_len": 768}
pt_full = {"name": "pretrain_t2t",      "size_gib": 7.71, "lines": 8_468_827, "max_len": 380}

# Sampling estimates based on actual MiniMind-tokenizer samples
# PT mini averages about 260 tokens per sample and loses almost nothing at length 768
# PT full also averages about 260, with about 82% efficiency at length 380

for d in [pt_mini, pt_full]:
    # Truncate the sampled mean of roughly 260 raw tokens at max_len
    avg_raw = 260
    avg_effective = min(avg_raw, d["max_len"])
    effective_b = avg_effective * d["lines"] / 1e9
    print(f"{d['name']}:")
    print(f"  Rows: {d['lines']/1e6:.1f}M, max_seq_len: {d['max_len']}, file: {d['size_gib']} GiB")
    print(f"  Sampled mean effective tokens per row: {avg_effective:.0f}")
    print(f"  Estimated total effective tokens: {effective_b:.2f}B")
    print()

print("=== Effective-token estimate for SFT, assistant labels only ===")
print()

sft_mini = {"name": "sft_t2t_mini", "size_gib": 1.62, "lines": 905_718,  "max_len": 768}
sft_full = {"name": "sft_t2t",      "size_gib": 13.13, "lines": 5_109_432, "max_len": 768}

# SFT samples contain roughly 55% prompt tokens and 45% assistant tokens
# About 34% of samples are truncated at length 768

for d in [sft_mini, sft_full]:
    avg_raw = 710          # Sampled mean prompt-plus-assistant token count
    trunc_rate = 0.34
    avg_trunc = avg_raw * (1 - trunc_rate) + d["max_len"] * trunc_rate
    avg_label = avg_trunc * 0.45     # Assistant share is 45%
    effective_b = avg_label * d["lines"] / 1e9
    print(f"{d['name']}:")
    print(f"  Rows: {d['lines']/1e6:.1f}M, max_seq_len: {d['max_len']}, file: {d['size_gib']} GiB")
    print(f"  Sampled mean assistant labels per row: {avg_label:.0f}")
    print(f"  Estimated total effective assistant tokens: {effective_b:.2f}B")
    print("  Note: prompt tokens occupy the sequence but do not contribute to loss")
    print()

print("Key distinction:")
print("  PT effective tokens are all post-truncation text tokens")
print("  SFT effective tokens are post-truncation assistant-label tokens, excluding the prompt")
print("  File size and row count are surface measures; effective tokens quantify the training signal.")


### 5.1 Tokens per Parameter

Chinchilla's 2022 compute-optimal estimate is roughly $Dpprox20N$ training Tokens for $N$ parameters. Later models often train far beyond this ratio because benchmarks continue improving and inference value may justify extra training.

For a 64M-parameter MiniMind model, about 1.82B Tokens in one epoch is roughly 28 Tokens/parameter; two epochs are about 57. This ratio is a reference, not a guarantee: data quality, repetition, architecture, and target deployment all matter.


### 5.2 Four Rules for Comparing Dataset Size

1. Use the same tokenizer; vocabulary and segmentation change Token counts.
2. Use the same `max_seq_len`; truncation changes effective Tokens.
3. Separate unique data from epoch count; repeating a small corpus is not new data.
4. For PT count supervised text Tokens; for SFT count assistant-label Tokens as well as total compute Tokens.

Comparable volume means effective loss Tokens under aligned tokenizer, length, and epoch assumptions—not GB or rows.


In [ ]:
# === Visualize the tokens-per-parameter reference frame ===
# The red line is the Chinchilla-optimal D=20N ratio; points above it train on more data
import matplotlib.pyplot as plt

# Each tuple gives label, parameter count, and PT token count from the preceding table
models = [
    ('MiniMind 64M (1 ep)', 64e6, 1.82e9),
    ('MiniMind 64M (2 ep)', 64e6, 3.64e9),
    ('SmolLM2 1.7B', 1.7e9, 11e12),
    ('Llama 3 8B', 8e9, 15e12),
    ('Chinchilla 70B', 70e9, 1.4e12),
    ('DeepSeek-V3 (37B active)', 37e9, 14.8e12),
]

N_line = np.logspace(6.5, 11.5, 100)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.plot(N_line, 20 * N_line, color='#dc2626', linestyle='--', linewidth=1.8,
        label='Chinchilla optimal: D = 20N')
offsets = {'Chinchilla 70B': (7, -12)}
for label, n_params, tokens in models:
    ax.scatter(n_params, tokens, s=46, color='#2563eb', zorder=3)
    ax.annotate(label, (n_params, tokens), textcoords='offset points',
                xytext=offsets.get(label, (7, 5)), fontsize=8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Parameters N')
ax.set_ylabel('Pretraining tokens D')
ax.set_title('Everyone trains far beyond the Chinchilla line')
ax.legend(loc='upper left', fontsize=8)
plt.show()

print("Key observation: every point on the red line satisfies D=20N; below is undertraining, above uses more data.")
print("  MiniMind at one epoch is near the line at 28 tokens/parameter; two epochs reaches 57")
print("  SmolLM2 and Llama 3 are far above it; giving small models more data has become a common post-2024 choice")


## Summary

Confirm you understand these (check in order):

1. ✅ Data sources: Common Crawl (main body) + Wikipedia + Books + Code + Papers
2. ✅ Pipeline five steps: text extraction -> language filtering -> quality filtering -> deduplication -> data mixing
3. ✅ HTML -> Text: remove script/style tags + remove HTML tags + clean whitespace
4. ✅ Quality filtering: heuristic rules (length/word-length/symbols) + PPL (language model scoring)
5. ✅ PII/safety filtering: text with normal quality may still need redaction or dropping due to emails, phone numbers, keys
6. ✅ Exact dedup: SHA256 hash, keep only one copy of completely identical items
7. ✅ MinHash: turn articles into fingerprints (n-gram -> hash -> take smallest K), similar fingerprints = similar articles
8. ✅ Decontamination: the training set and benchmark must not highly overlap, otherwise benchmark scores are untrustworthy
9. ✅ Data mixing: upgrade from "high quality more epochs" to an evaluable data recipe
10. ✅ Common benchmarks: MMLU/C-Eval for knowledge, GSM8K/MATH for math, HumanEval/MBPP for code, BBH/GPQA for hard-problem reasoning
11. ✅ Evaluation tools: lm-eval, EvalScope, OpenCompass, OpenAI Evals are responsible for running benchmarks reproducibly
12. ✅ Industrial tools: Data-Juicer, DataTrove, NeMo Curator, IBM Data Prep Kit all help you combine and run data operators
13. ✅ After Tokenize: concatenate into a "token noodle" -> cut into fixed-length chunks -> start training
14. ✅ FIM (Fill-in-the-Middle): rearrange text with <PRE>/<SUF>/<MID> to train code completion ability

15. ✅ Post-training data is 70%+ synthetic: real annotation cost cannot sustain iteration speed
16. ✅ Self-Instruct: 175 seeds -> 52K instructions, cost < $500
17. ✅ Evol-Instruct: three directions (depth/breadth/eliminate) systematically expand the difficulty distribution
18. ✅ Distillation: Alpaca spent $600 on API calls to approach GPT-3.5; the compliance constraint is "cannot use competing APIs"
19. ✅ STaR: only keep reasoning traces of correct answers as training data, self-improving over multiple iterations
20. ✅ Data flywheel risks: mode collapse, error amplification, distribution drift; AI + human mix is the industrial mainstream

**One-line summary**: data engineering is one of the most critical stages of LLM training. The teaching-version pipeline helps you see the main thread; the key of the industrial-version pipeline is connecting cleaning, redaction, deduplication, decontamination, mixing, and evaluation into a closed loop, and iterating on the data recipe continuously.

## Exercises

The 3 small exercises below are all "fill in the blank + assert check". Try to think for 3 minutes first; you can ask AI for ideas, break down steps, or check direction, but it is not recommended to have AI produce the complete answer directly.


In [ ]:
# Exercise 1: Complete an email redaction function
# Hint: you can use re.sub to replace matched emails with <EMAIL>.

import re

def redact_email(text):
    """Replace emails in the text with <EMAIL>"""
    email_pattern = r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"
    # TODO: change the None below to a single line of re.sub(...)
    result = None
    return result

# After filling in, uncomment the next line and run the check
# assert redact_email("mail me at a@b.com") == "mail me at <EMAIL>"
# print("Exercise 1 passed: you can do the most basic PII redaction.")


In [ ]:
# Exercise 2: Complete Jaccard similarity
# Hint: intersection uses &, union uses |.

def jaccard_similarity(a, b):
    """Compute the Jaccard similarity of two sets"""
    # TODO: change the None below to len(intersection) / len(union)
    score = None
    return score

# After filling in, uncomment the three lines below and run the check
# s1 = {"the cat", "cat sat", "sat down"}
# s2 = {"the cat", "cat sat", "sat here"}
# assert abs(jaccard_similarity(s1, s2) - 0.5) < 1e-9
# print("Exercise 2 passed: you can compute the core similarity of approximate dedup.")


In [ ]:
# Exercise 3: Complete the weighted sampling amount of a data recipe
# Hint: effective sampling amount = raw token count * sampling weight.

sources = [
    {"name": "web", "tokens": 1000, "weight": 0.5},
    {"name": "wiki", "tokens": 100, "weight": 2.0},
    {"name": "code", "tokens": 200, "weight": 1.5},
]

# TODO: change None to a list comprehension that computes the effective sampling amount for each source
sampled_tokens = None

# After filling in, uncomment the two lines below and run the check
# assert sampled_tokens == [500, 200, 300]
# print("Exercise 3 passed: you understand sampling weights in a data recipe.")


## References

- T5 / C4 — cleaned Common Crawl
- Textbooks Are All You Need — high-quality synthetic textbook-style data
- The Llama 3 Herd of Models — large-scale filtering, deduplication, and mixing
- FineWeb / FineWeb-Edu — Common Crawl ablations and educational scoring
- Self-Instruct — seed instructions expanded by a model
- WizardLM — Evol-Instruct
- STaR — verified self-generated reasoning
- LIMA — alignment with 1,000 curated examples
- SemDeDup — semantic deduplication at web scale
- Data-Juicer, DataTrove, NeMo Curator — open data-processing systems
